# API vulnerabilities  (MODEL SOLUTION, instructor copy, do not distribute)

# Coding problem, 5.1.c: Detect broken object-level authorization (BOLA)

**Skill:** Analyze Risk · **Maps to:** 5.1.B · **Language:** Python 3 (pytest)

## Problem
Build the server-side authorization check an API gateway should run before returning an object.
Given the authenticated caller's user id, the id of the object they are requesting, an
object-ownership map, and the caller's role, decide whether the request is allowed.

Rules:
- A request is **allowed** if the caller owns the requested object (the object's `owner` equals
  the caller's `user_id`), OR the caller's `role` is `"admin"`.
- A request is **denied** (a BOLA violation) if the object exists but the caller neither owns it
  nor is an admin.
- A request for an object id that does not exist in the ownership map is **denied** with reason
  `"not_found"`.

## Function contract
Implement `solve(request: dict, ownership: dict) -> dict` in the code editor (see the Reference section below).

## Input
- `request`: `{"user_id": str, "role": str, "object_id": str}`
- `ownership`: `dict` mapping `object_id -> owner user_id` (e.g. `{"482": "u1", "483": "u2"}`)

## Output
A dict: `{"allowed": bool, "reason": str}` where `reason` is one of
`"owner"`, `"admin"`, `"not_owner"`, `"not_found"`.

## Examples
```
request={"user_id":"u1","role":"user","object_id":"482"}, ownership={"482":"u1"}
    -> {"allowed": True, "reason": "owner"}

request={"user_id":"u1","role":"user","object_id":"483"}, ownership={"483":"u2"}
    -> {"allowed": False, "reason": "not_owner"}

request={"user_id":"u9","role":"admin","object_id":"483"}, ownership={"483":"u2"}
    -> {"allowed": True, "reason": "admin"}

request={"user_id":"u1","role":"user","object_id":"999"}, ownership={"483":"u2"}
    -> {"allowed": False, "reason": "not_found"}
```

## Constraints
- Deterministic, offline, standard library only.
- Must pass all cases in the tests (visible + hidden).

## Reference
Any implementation that satisfies this interface can be swapped in (reference, student, or
composite) and must pass the tests unchanged.

### Entry point
```python
def solve(request: dict, ownership: dict) -> dict
```

### Input
- `request`: `{"user_id": str, "role": str, "object_id": str}`, the authenticated caller and the
  object they are requesting.
- `ownership`: `dict[str, str]` mapping `object_id -> owner user_id`.

### Output (`result`)
`{"allowed": bool, "reason": str}` where `reason` is exactly one of:
- `"owner"`, caller's `user_id` matches the object's owner.
- `"admin"`, caller's `role` is `"admin"` (allowed regardless of ownership).
- `"not_owner"`, object exists but caller neither owns it nor is admin.
- `"not_found"`, `object_id` is not a key in `ownership`.

### Guarantees / rules
- Pure function, deterministic, no I/O, no network. Standard library only.
- Ownership check takes priority: if the caller owns the object, reason is `"owner"` even if role
  is also `"admin"`.
- `"not_found"` is checked independently of role/ownership, a missing object is always denied.

### Acceptance
Your code passes when every test below shows a checkmark (some are hidden until you run them).


In [ ]:
# Reference solution
"""Reference solution for 5.1.c. MUST pass 100% of the tests.
Also the default 'blackbox' implementation students can swap out (see the hidden grading tests)."""
from typing import Dict


def solve(request: Dict[str, str], ownership: Dict[str, str]) -> Dict[str, object]:
    """Decide whether an API request is authorized (BOLA check)."""
    object_id = request.get("object_id")
    user_id = request.get("user_id")
    role = request.get("role")

    if object_id not in ownership:
        return {"allowed": False, "reason": "not_found"}

    owner = ownership[object_id]
    if owner == user_id:
        return {"allowed": True, "reason": "owner"}
    if role == "admin":
        return {"allowed": True, "reason": "admin"}
    return {"allowed": False, "reason": "not_owner"}


## Self-check tests
Run the cell below to check your `solve` implementation above.

In [ ]:
import __main__ as impl  # the functions defined above

# ---- visible cases -------------------------------------------------------
def test_owner_allowed():
    req = {"user_id": "u1", "role": "user", "object_id": "482"}
    own = {"482": "u1"}
    assert impl.solve(req, own) == {"allowed": True, "reason": "owner"}


def test_non_owner_denied():
    req = {"user_id": "u1", "role": "user", "object_id": "483"}
    own = {"483": "u2"}
    assert impl.solve(req, own) == {"allowed": False, "reason": "not_owner"}


def test_admin_allowed():
    req = {"user_id": "u9", "role": "admin", "object_id": "483"}
    own = {"483": "u2"}
    assert impl.solve(req, own) == {"allowed": True, "reason": "admin"}


def test_not_found():
    req = {"user_id": "u1", "role": "user", "object_id": "999"}
    own = {"483": "u2"}
    assert impl.solve(req, own) == {"allowed": False, "reason": "not_found"}


# ---- hidden / edge cases -------------------------------------------------
def test_owner_priority_over_admin_role():
    req = {"user_id": "u1", "role": "admin", "object_id": "482"}
    own = {"482": "u1"}
    assert impl.solve(req, own) == {"allowed": True, "reason": "owner"}


def test_not_found_checked_before_role():
    req = {"user_id": "u1", "role": "admin", "object_id": "does_not_exist"}
    own = {"482": "u1"}
    assert impl.solve(req, own) == {"allowed": False, "reason": "not_found"}


def test_empty_ownership():
    req = {"user_id": "u1", "role": "user", "object_id": "482"}
    assert impl.solve(req, {}) == {"allowed": False, "reason": "not_found"}


# ---- run every test above ------------------------------------------------
_results = []
for _name, _fn in list(globals().items()):
    if _name.startswith("test_") and callable(_fn):
        try:
            _fn()
            _results.append((_name, None))
        except Exception as _e:
            _results.append((_name, _e))
for _name, _e in _results:
    print(("PASS  " if _e is None else "FAIL  ") + _name + ("" if _e is None else f": {_e!r}"))
print(f"{sum(e is None for _, e in _results)}/{len(_results)} tests passed")
